In [7]:
import pandas as pd
import numpy as np

# Tải dữ liệu từ file CSV bạn đã cung cấp
df = pd.read_excel('Online Retail.xlsx')

# 1. Làm sạch dữ liệu: Xử lý giá trị thiếu và loại bỏ số lượng âm [cite: 9]
df = df.dropna(subset=['CustomerID']) 
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 2. Tạo biến mục tiêu: Doanh số (Target Variable) [cite: 6]
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

# 3. Kỹ thuật đặc trưng (Feature Engineering) [cite: 13, 17]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.month

# Lấy mẫu nhỏ (ví dụ 1000 dòng) để thuật toán tự viết chạy nhanh hơn [cite: 53]
df_sample = df.sample(1000, random_state=42) 

product_sales = df_sample.groupby(['StockCode', 'Month']).agg({
    'TotalSales': 'sum',
    'UnitPrice': 'mean'
}).reset_index()

# Chuyển đổi biến phân loại thành số [cite: 10]
product_sales['StockCode_ID'] = pd.factorize(product_sales['StockCode'])[0]

# Định nghĩa X (Features) và y (Target) [cite: 6, 14]
X = product_sales[['StockCode_ID', 'Month', 'UnitPrice']]
y = product_sales['TotalSales']

X_np = X.values
y_np = y.values

In [8]:
def calculate_mse(y):
    if len(y) == 0: return 0
    return np.mean((y - np.mean(y))**2)

class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth

    def fit(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) <= 5:
            return np.mean(y)

        best_mse = float('inf')
        best_split = None
        
        for f_idx in range(X.shape[1]):
            # Tối ưu: Lấy các giá trị đại diện để làm ngưỡng chia [cite: 20]
            thresholds = np.percentile(X[:, f_idx], [25, 50, 75])
                
            for thresh in thresholds:
                idx_l = X[:, f_idx] <= thresh
                idx_r = ~idx_l
                
                if sum(idx_l) == 0 or sum(idx_r) == 0: continue
                
                mse = calculate_mse(y[idx_l]) + calculate_mse(y[idx_r])
                if mse < best_mse:
                    best_mse = mse
                    best_split = (f_idx, thresh, idx_l, idx_r)

        if not best_split: return np.mean(y)

        f_idx, thresh, il, ir = best_split
        left = self.fit(X[il], y[il], depth + 1)
        right = self.fit(X[ir], y[ir], depth + 1)
        return (f_idx, thresh, left, right)

    def predict_row(self, row, node):
        if not isinstance(node, tuple): return node
        f_idx, thresh, left, right = node
        return self.predict_row(row, left if row[f_idx] <= thresh else right)

class RandomForestCustom:
    def __init__(self, n_trees=5, max_depth=3):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        for _ in range(self.n_trees):
            # Lấy mẫu ngẫu nhiên (Bootstrapping) 
            idx = np.random.choice(len(X), len(X), replace=True)
            tree_obj = DecisionTree(max_depth=self.max_depth)
            tree_data = tree_obj.fit(X[idx], y[idx])
            self.trees.append((tree_obj, tree_data))

    def predict(self, X):
        all_preds = []
        for tree_obj, tree_data in self.trees:
            all_preds.append([tree_obj.predict_row(row, tree_data) for row in X])
        return np.mean(all_preds, axis=0)

In [ ]:
# Huấn luyện mô hình 
rf = RandomForestCustom(n_trees=5, max_depth=3)
rf.fit(X_np, y_np)

# Dự đoán 
y_pred = rf.predict(X_np)

# 1. Chỉ số Hồi quy 
mae = np.mean(np.abs(y_np - y_pred))
rmse = np.sqrt(np.mean((y_np - y_pred)**2))
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

# 2. Chỉ số Phân loại (Accuracy & Confusion Matrix)
# Chia doanh số thành 2 nhóm: Cao (1) và Thấp (0) dựa trên trung vị
threshold = np.median(y_np)
y_true_bin = (y_np > threshold).astype(int)
y_pred_bin = (y_pred > threshold).astype(int)

# Tính Accuracy
accuracy = np.mean(y_true_bin == y_pred_bin)
print(f"Accuracy (Phân loại doanh số): {accuracy*100:.2f}%")

# Tính Confusion Matrix thủ công
tp = np.sum((y_true_bin == 1) & (y_pred_bin == 1))
tn = np.sum((y_true_bin == 0) & (y_pred_bin == 0))
fp = np.sum((y_true_bin == 0) & (y_pred_bin == 1))
fn = np.sum((y_true_bin == 1) & (y_pred_bin == 0))

print("\nConfusion Matrix:")
print(f"PN\\PD | Thấp | Cao")
print(f"Thấp  |  {tn}  |  {fp}")
print(f"Cao   |  {fn}  |  {tp}")

MAE: 23.09
RMSE: 116.03
Accuracy (Phân loại doanh số): 50.58%

Confusion Matrix:
PN\PD | Thấp | Cao
Thấp  |  8  |  468
Cao   |  2  |  473
